# Day 01 — Descriptive statistics (200 bệnh nhân)

**Slides (tải về):** [`day01_slides.pptx`](_static/slides/day01_slides.pptx)

Mục tiêu của buổi này:
- Đọc dữ liệu CSV và hiểu cấu trúc cohort.
- Làm Table 1 tổng quan và theo nhóm EGFR.
- Vẽ 3 biểu đồ mô tả cơ bản.

Sản phẩm cuối buổi:
- 1 Table 1.
- 3 biểu đồ.
- 5 dòng nhận xét ngắn.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (6,4)


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Nếu chạy trên Colab và không thấy file, notebook sẽ tải demo CSV từ GitHub.
GITHUB_USER = "YOUR_GITHUB_USERNAME"
REPO_NAME = "YOUR_REPO_NAME"
BRANCH = "main"

def download_from_github(rel_path: str) -> Path:
    import urllib.request
    url = f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO_NAME}/{BRANCH}/{rel_path}"
    out_path = Path(rel_path).name
    print("Tải file:", url)
    urllib.request.urlretrieve(url, out_path)
    return Path(out_path)

def load_csv(rel_path: str, fallback_upload: bool = True) -> pd.DataFrame:
    candidates = [
        Path(rel_path),
        Path("../")/rel_path,
        Path("../../")/rel_path,
        Path("data")/Path(rel_path).name,
        Path("_static/data")/Path(rel_path).name,
        Path("../data")/Path(rel_path).name,
    ]
    for p in candidates:
        if p.exists():
            return pd.read_csv(p)

    # Thử tải từ GitHub
    try:
        p = download_from_github(rel_path)
        return pd.read_csv(p)
    except Exception as e:
        print("Không tải được từ GitHub:", e)

    # Cuối cùng: upload thủ công
    if fallback_upload:
        try:
            from google.colab import files  # type: ignore
            uploaded = files.upload()
            if len(uploaded) > 0:
                up_path = Path(list(uploaded.keys())[0])
                return pd.read_csv(up_path)
        except Exception:
            pass

    raise FileNotFoundError("Không tìm thấy CSV. Cần clone repo hoặc upload file.")

df = load_csv("data/nsclc_egfr_radiomics_simplified.csv")
df.head()


## 1) Nhìn nhanh dataset

In [ ]:
df.shape, df.columns.tolist()

In [ ]:
df['egfr_mutation'].value_counts(normalize=True).rename('tỉ lệ').to_frame()

## 2) Table 1 đơn giản
Table 1 gồm: Overall, EGFR(0), EGFR(1).

Gợi ý: buổi này làm bản đơn giản, không cần quá nhiều biến.

In [ ]:
def summarize_cont(series: pd.Series):
    series = series.dropna()
    return {
        "n": int(series.shape[0]),
        "mean": float(series.mean()),
        "std": float(series.std(ddof=1)),
        "median": float(series.median()),
        "q1": float(series.quantile(0.25)),
        "q3": float(series.quantile(0.75)),
    }

def summarize_cat(series: pd.Series):
    s = series.fillna("NA").astype(str)
    counts = s.value_counts()
    total = counts.sum()
    top = counts.head(6)
    return {k: f"{v} ({v/total:.1%})" for k, v in top.items()}

cont_vars = ["age", "tumor_size_mm", "tumor_volume_cm3"]
cat_vars = ["sex", "smoking_status", "histology", "stage"]

rows = []
for var in cont_vars:
    overall = summarize_cont(df[var])
    g0 = summarize_cont(df.loc[df["egfr_mutation"]==0, var])
    g1 = summarize_cont(df.loc[df["egfr_mutation"]==1, var])
    rows.append([var,
                 f'{overall["median"]:.1f} [{overall["q1"]:.1f}; {overall["q3"]:.1f}]',
                 f'{g0["median"]:.1f} [{g0["q1"]:.1f}; {g0["q3"]:.1f}]',
                 f'{g1["median"]:.1f} [{g1["q1"]:.1f}; {g1["q3"]:.1f}]'])
table1_cont = pd.DataFrame(rows, columns=["Biến", "Overall", "EGFR=0", "EGFR=1"])

table1_cont


In [ ]:
# Biến định tính: in nhanh top nhóm
for var in cat_vars:
    print("\n---", var, "---")
    print("Overall:", summarize_cat(df[var]))
    print("EGFR=0 :", summarize_cat(df.loc[df["egfr_mutation"]==0, var]))
    print("EGFR=1 :", summarize_cat(df.loc[df["egfr_mutation"]==1, var]))


## 3) Biểu đồ mô tả cơ bản

In [ ]:
# Histogram tuổi
plt.hist(df["age"].dropna(), bins=15)
plt.title("Histogram: age")
plt.xlabel("age")
plt.ylabel("count")
plt.show()


In [ ]:
# Boxplot tumor_size_mm theo EGFR
data0 = df.loc[df["egfr_mutation"]==0, "tumor_size_mm"].dropna()
data1 = df.loc[df["egfr_mutation"]==1, "tumor_size_mm"].dropna()
plt.boxplot([data0, data1], labels=["EGFR=0", "EGFR=1"])
plt.title("tumor_size_mm theo EGFR")
plt.ylabel("tumor_size_mm")
plt.show()


In [ ]:
# Bar chart: tỉ lệ EGFR theo sex
ct = pd.crosstab(df["sex"], df["egfr_mutation"], normalize="index")
ct.plot(kind="bar")
plt.title("Tỉ lệ EGFR theo sex")
plt.ylabel("Tỉ lệ")
plt.legend(title="EGFR")
plt.show()


## Bài tập cuối buổi
1) EGFR(1) chiếm bao nhiêu phần trăm?
2) Tuổi nên báo cáo theo mean hay median? Lý do.
3) Nhìn boxplot tumor_size_mm: hai nhóm khác nhau nhiều không?
4) Ghi 5 dòng nhận xét ngắn về cohort.